## 12.10 转置卷积


### 环境配置


In [3]:
import os, sys
sys.path.insert(0, "..")
os.environ["TILE_FWK_DEVICE_ID"] = "1"
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    import pypto
    import torch
    import torch_npu
    import torchvision
    from torch import nn
    from torch.nn import functional as F
    from src.D2LFunction import *

import logging
# pypto 导入会把根 logger 调到 DEBUG，压回 WARNING 避免刷屏
logging.getLogger().setLevel(logging.WARNING)
logging.getLogger("matplotlib").setLevel(logging.WARNING)
logging.getLogger("PIL").setLevel(logging.WARNING)

device_id = int(os.environ["TILE_FWK_DEVICE_ID"])
torch.npu.set_device(device_id)
device = f"npu:{device_id}"


---


### 练习 12.10.1

**题目：** 在 [12.10.3节](./12.10_transposed_conv.ipynb)中，卷积输入`X`和转置的卷积输出`Z`具有相同的形状。他们的数值也相同吗？为什么？

**解答：**

&emsp;&emsp;数值通常不同。

1. **形状相同**：例中$3\times 3$的卷积输出$Y$再经过转置卷积（步幅1、无填充）得到$3\times 3$的$Z$，恰好与原始输入$X$同形状。这是由转置卷积是普通卷积的“逆操作”这一性质决定的，输出形状公式为$n_h'=(n_h-k_h+2p_h)/s_h+1$，这里$n_h=2$，$k_h=2$，$p_h=0$，$s_h=1$，故$n_h'=3$。
2. **数值不同**：$Z$是$Y$经过转置卷积核的“广播”得到的，其每个位置都是$Y$中多个元素乘以卷积核权重再累加的结果，与原始输入$X$没有一一对应关系，因此数值一般不同。

以下使用 `torch` 编程进行验证：


In [6]:
def trans_conv(X, K):
    h, w = K.shape
    Y = torch.zeros((X.shape[0] + h - 1, X.shape[1] + w - 1))
    for i in range(X.shape[0]):
        for j in range(X.shape[1]):
            Y[i: i + h, j: j + w] += X[i, j] * K
    return Y

X = torch.arange(9.0).reshape(3, 3)
K = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
Y = corr2d(X, K)          # 普通卷积（互相关）
Z = trans_conv(Y, K)      # 转置卷积
print('X.shape =', X.shape, ', Z.shape =', Z.shape)
print('形状相同:', X.shape == Z.shape)
print('数值相同:', torch.equal(X, Z))

X.shape = torch.Size([3, 3]) , Z.shape = torch.Size([3, 3])
形状相同: True
数值相同: False


In [7]:
# 使用高级API验证（与正文一致）：Y(2,2) 经转置卷积后输出 (3,3)
X4, K4 = Y.reshape(1, 1, 2, 2), K.reshape(1, 1, 2, 2)
tconv = nn.ConvTranspose2d(1, 1, kernel_size=2, bias=False)
tconv.weight.data = K4
Z_api = tconv(X4).reshape(3, 3)
print('高级API与手工实现一致:', torch.allclose(Z_api, Z))

高级API与手工实现一致: True


&emsp;&emsp;从结果可以看到，$X$与$Z$形状相同（都是$3\times 3$），但数值完全不同。

使用 `PyPTO` 编程进行验证（与矩阵变换的联系）：
&emsp;&emsp;转置卷积等价于“卷积的稀疏权重矩阵$W$的转置$W^\top$做矩阵乘法”。本教程在 [6.2节](../06_pypto_convolutional_networks/06.02_conv_layer.ipynb) 与 `src/PyPTOConv2DModule.py` 中，正是用 `pypto.matmul` 的 im2col 路线实现卷积（普通卷积用 $C=\text{cols}\cdot K^\top$，其转置对应 $K\cdot \text{cols}^\top$）。下面用 PyPTO `matmul` 验证转置卷积的矩阵乘法等价性：

&emsp;&emsp;注：该用例矩阵很小（9×4 @ 4×1），远低于 Cube 的典型 tile 尺寸，此处仅验证数学等价性，不体现性能；`_matmul_bt_kernel` 的 tile 配置（`set_cube_tile_shapes([128,128],[64,256],[256,256])`）来自性能调优文档针对常规 matmul 尺寸的推荐值。


In [9]:
# 用稀疏权重矩阵W的转置实现转置卷积（对应正文 kernel2matrix 的 W.T）
def kernel2matrix(K):
    k, W = torch.zeros(5), torch.zeros((4, 9))
    k[:2], k[3:5] = K[0, :], K[1, :]
    W[0, :5], W[1, 1:6], W[2, 3:8], W[3, 4:] = k, k, k, k
    return W

W = kernel2matrix(K)
Z_matmul = torch.matmul(W.T, Y.reshape(-1)).reshape(3, 3)
print('W^T @ vec(Y) 与转置卷积一致:', torch.allclose(Z_matmul, Z))

W^T @ vec(Y) 与转置卷积一致: True


In [10]:
# 转置卷积: Z = W^T @ vec(Y)
# 复用 src/PyPTOConv2DModule 中同款 PyPTO matmul kernel（Cube tile 配置见性能调优文档推荐值）
from src.PyPTOConv2DModule import _matmul_bt_kernel

# C = A @ B^T, 取 A = W^T(9,4), B = vec(Y)^T(1,4), b_trans=True
A = W.T.float().contiguous().to(device)                   # (9, 4)
B = Y.reshape(-1, 1).t().contiguous().to(device)          # (1, 4)
out = torch.zeros(9, 1, device=device)
_matmul_bt_kernel(A, B, out)
Z_pt = out.reshape(3, 3).cpu()
print('PyPTO matmul 验证转置卷积等价性:', torch.allclose(Z_pt, Z, atol=1e-5))

PyPTO matmul 验证转置卷积等价性: True


---


### 练习 12.10.2

**题目：** 使用矩阵乘法来实现卷积是否有效率？为什么？

**解答：**

&emsp;&emsp;直接使用稀疏权重矩阵（如正文中的$W$）做矩阵乘法并不是最有效的实现方式：

1. **稀疏性丧失**：卷积核是局部连接的，每个输出位置仅与输入的局部区域相连，稀疏权重矩阵中大部分元素为0；直接做矩阵乘法会引入大量零元素的无效计算，计算与存储效率低。
2. **权值共享未利用**：卷积核的权值在整个输入上共享，相同权值被反复使用；而稀疏矩阵乘法中每个非零元素仍需单独参与乘法累加，无法体现权值共享的复算优势。
3. **核尺寸增大时复杂度急剧上升**：权重矩阵规模随核尺寸和输入尺寸乘积增长，大尺寸卷积核场景下不适用。

&emsp;&emsp;实践中更高效的做法是 **im2col + 稠密矩阵乘法**（如本教程 `PyPTOConv2DModule.py` 的做法）：先用 `torch.as_strided` 零拷贝地取出每个卷积窗口，拼成 (M, K) 的 im2col 矩阵，再调用 `pypto.matmul`（Cube 单元，FP32）一次完成所有窗口的卷积。该路线把卷积变成了稠密GEMM，能充分发挥昇腾Cube算力；权重矩阵$K$（而非$W$）只有 $C_{in}\times k_h\times k_w$ 个元素，稀疏性和权值共享问题自然消解。

---
## 参考答案来源
参考答案和 PyTorch 代码实现来源：[https://datawhalechina.github.io/d2l-ai-solutions-manual/#/](https://datawhalechina.github.io/d2l-ai-solutions-manual/#/)
